# Atlas Weather Intelligence Platform
## Data Science Notebook — Global Weather Repository Analysis

**PM Accelerator AI Engineer Internship — Dual Role Submission**

---

### PM Accelerator Mission
> *PM Accelerator is the world's most accessible product management program, providing aspiring and experienced PMs with the skills, mentorship, and real-world experience needed to land and excel in product management roles.*

---

### What this notebook covers
| Section | Assessment Requirement |
|---|---|
| 1. Data Loading & Inspection | D1 |
| 2. Data Cleaning | D2, D3, D4 |
| 3. Exploratory Data Analysis | D5, D6, D7 |
| 4. Anomaly Detection | D11 |
| 5. Forecasting — Baseline Model | D8, D9, D10 |
| 6. Forecasting — Gradient Boosting | D12 |
| 7. Ensemble Model | D13 |
| 8. Seasonal Pattern Analysis | D14 |
| 9. Environmental Impact Analysis | D15 |
| 10. Geographic Patterns | D17, D18 |
| 11. Per-City Forecast Comparison | own addition |
| 12. Model Artifact Export | production serving |

**Dataset:** [Global Weather Repository — Kaggle](https://www.kaggle.com/datasets/nelgiriyewithana/global-weather-repository)

*Built by Maryam Shanabli*

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys, pathlib
sys.path.insert(0, str(pathlib.Path().absolute()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')
print('Libraries loaded.')

## 1. Data Loading & Inspection

In [ ]:
df_raw = pd.read_csv('data/GlobalWeatherRepository.csv')
print(f'Shape: {df_raw.shape}')
print(f'Columns ({len(df_raw.columns)}):', df_raw.columns.tolist())
df_raw.head(3)

In [ ]:
print('Missing values per column:')
nulls = df_raw.isna().sum()
print(nulls[nulls > 0] if nulls.any() else 'NONE — zero missing values across all 41 columns')
print()
print('Duplicates:', df_raw.duplicated().sum())
print('last_updated sample:', df_raw['last_updated'].head(3).tolist())

**Finding:** Zero missing values, near-zero duplicates. The dataset is clean at the structural level.
Cleaning effort focuses on two real problems found during inspection:
1. Physically impossible values (temperature 79°C, wind 2963 km/h, pressure 3006 mb)
2. Country name contamination — same physical location logged under multiple language translations

## 2. Data Cleaning

In [ ]:
from data_cleaning import clean_dataset

df_clean = clean_dataset(df_raw.copy())
print(f'Countries before canonicalization: {df_raw["country"].nunique()}')
print(f'Countries after canonicalization:  {df_clean["country_clean"].nunique()}')
print(f'Physically implausible readings:   {df_clean["any_implausible_reading"].sum()}')
df_clean[df_clean['any_implausible_reading']][['location_name','temperature_celsius','wind_kph','pressure_mb']]

In [ ]:
limits = {'temperature_celsius': (-90, 60), 'wind_kph': (0, 410), 'pressure_mb': (850, 1090), 'humidity': (0, 100)}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (col, (lo, hi)) in zip(axes, limits.items()):
    clean = df_clean[~df_clean['any_implausible_reading']]
    ax.hist(clean[col].dropna(), bins=50, color='#2d6a9f', alpha=0.8)
    ax.axvline(hi, color='red', linestyle='--', linewidth=1, label=f'Limit: {hi}')
    ax.set_title(col.replace('_', ' ').title())
    ax.legend(fontsize=8)
plt.suptitle('Distribution of key fields after removing implausible readings', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

df_clean_norm = df_clean[~df_clean['any_implausible_reading']].copy()
scaler = StandardScaler()
df_clean_norm[['temp_scaled', 'humidity_scaled']] = scaler.fit_transform(
    df_clean_norm[['temperature_celsius', 'humidity']]
)
print('Normalized temperature — mean:', round(df_clean_norm['temp_scaled'].mean(), 4),
      'std:', round(df_clean_norm['temp_scaled'].std(), 4))
df_clean = df_clean_norm  # use the normalized, implausible-filtered frame from here on
print('Working dataset shape:', df_clean.shape)

## 3. Exploratory Data Analysis

In [ ]:
df_clean['date'] = df_clean['last_updated'].dt.date
daily_temp = df_clean.groupby('date')['temperature_celsius'].mean().reset_index()
daily_temp['date'] = pd.to_datetime(daily_temp['date'])
daily_temp['rolling_30d'] = daily_temp['temperature_celsius'].rolling(30).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_temp['date'], daily_temp['temperature_celsius'], alpha=0.3, color='#2d6a9f', linewidth=0.8, label='Daily mean')
ax.plot(daily_temp['date'], daily_temp['rolling_30d'], color='#e07b54', linewidth=2, label='30-day rolling mean')
ax.set_title('Global mean temperature over time (last_updated as time index — D9)', fontweight='bold')
ax.set_ylabel('Temperature (°C)')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Global mean temp: {df_clean['temperature_celsius'].mean():.2f}°C")
print(f"Temp/humidity correlation: {df_clean['temperature_celsius'].corr(df_clean['humidity']):.3f}")

In [ ]:
precip_nonzero = df_clean[df_clean['precip_mm'] > 0]['precip_mm']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_clean['precip_mm'], bins=60, color='#2d6a9f', alpha=0.8)
axes[0].set_title('Precipitation — all readings')
axes[1].hist(precip_nonzero, bins=60, color='#5ba4cf', alpha=0.8)
axes[1].set_title('Precipitation — non-zero only')
plt.suptitle('Precipitation is right-skewed and zero-inflated', fontweight='bold')
plt.tight_layout()
plt.show()
print(f"% readings with zero precipitation: {(df_clean['precip_mm']==0).mean()*100:.1f}%")

In [ ]:
corr_cols = ['temperature_celsius','humidity','precip_mm','wind_kph','pressure_mb','uv_index','air_quality_PM2.5','air_quality_us-epa-index']
corr = df_clean[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, linewidths=0.5)
ax.set_title('Correlation matrix — weather & air quality features', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
city_temp = df_clean.groupby('location_name')['temperature_celsius'].mean().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
city_temp.tail(10).plot.barh(ax=axes[0], color='#e07b54')
axes[0].set_title('10 hottest cities (mean temp °C)')
city_temp.head(10).plot.barh(ax=axes[1], color='#2d6a9f')
axes[1].set_title('10 coldest cities (mean temp °C)')
plt.suptitle('Geographic temperature extremes', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Anomaly Detection (Advanced — D11)

In [ ]:
from anomaly_detection import build_location_baselines

baselines = build_location_baselines(df_clean)

df_with_base = df_clean.merge(
    baselines[['location_name','country_clean','temp_mean','temp_std']],
    on=['location_name','country_clean'], how='left'
)
df_with_base['temp_z'] = (df_with_base['temperature_celsius'] - df_with_base['temp_mean']) / df_with_base['temp_std'].replace(0, np.nan)
stat_anomalies = df_with_base[df_with_base['temp_z'].abs() > 2.5]
print('Physically impossible readings: 4 (already removed from df_clean above)')
print(f'Statistically unusual (z>2.5):  {len(stat_anomalies)}')
print(f'As % of dataset:               {len(stat_anomalies)/len(df_clean)*100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(df_with_base['last_updated'], df_with_base['temp_z'], alpha=0.05, s=1, color='#2d6a9f', label='Normal')
ax.scatter(stat_anomalies['last_updated'], stat_anomalies['temp_z'], alpha=0.6, s=10, color='#e07b54', label='Anomalous (|z|>2.5)')
ax.axhline(2.5, color='red', linestyle='--', linewidth=1)
ax.axhline(-2.5, color='red', linestyle='--', linewidth=1)
ax.set_title('Temperature anomalies over time — z-score per location baseline', fontweight='bold')
ax.set_ylabel('Z-score vs location historical mean')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Forecasting — Seasonal Naive Baseline (D8, D9, D10)

In [ ]:
from forecasting import seasonal_naive_predict, train_gradient_boosting, evaluate, add_time_features

cutoff = df_clean['last_updated'].max() - pd.Timedelta(days=60)
train_df = df_clean[df_clean['last_updated'] <= cutoff]
test_df  = df_clean[df_clean['last_updated'] > cutoff]
print(f'Train: {len(train_df):,} rows | Test: {len(test_df):,} rows')

naive_preds = seasonal_naive_predict(train_df, test_df)
naive_metrics = evaluate(test_df['temperature_celsius'].values, naive_preds)
print(f'Baseline (seasonal naive) — MAE: {naive_metrics["mae"]:.3f}°C | RMSE: {naive_metrics["rmse"]:.3f}°C')

## 6. Forecasting — Gradient Boosting Model (D12)

In [ ]:
model, features = train_gradient_boosting(train_df)
test_feat = add_time_features(test_df)
gb_preds = model.predict(test_feat[features])
gb_metrics = evaluate(test_df['temperature_celsius'].values, gb_preds)

print(f'Gradient Boosting — MAE: {gb_metrics["mae"]:.3f}°C | RMSE: {gb_metrics["rmse"]:.3f}°C')
improvement = (1 - gb_metrics['mae']/naive_metrics['mae']) * 100
print(f'Improvement over baseline: {improvement:.1f}% MAE reduction')
for f, imp in zip(features, model.feature_importances_):
    print(f'  {f}: {imp:.3f}')

In [ ]:
feat_imp = pd.Series(model.feature_importances_, index=features).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
feat_imp.plot.barh(ax=ax, color='#2d6a9f')
ax.set_title('Feature importances — Gradient Boosting model', fontweight='bold')
plt.tight_layout()
plt.show()
print('Latitude dominates — climate zone determines temperature more than season or longitude.')

## 7. Ensemble Model (D13)

Combines the seasonal-naive baseline and the gradient boosting model. Two weighting strategies are evaluated rather than assumed.

In [ ]:
ensemble_avg = (naive_preds + gb_preds) / 2
ensemble_weighted = 0.3 * naive_preds + 0.7 * gb_preds

ensemble_avg_metrics = evaluate(test_df['temperature_celsius'].values, ensemble_avg)
ensemble_weighted_metrics = evaluate(test_df['temperature_celsius'].values, ensemble_weighted)

comparison = pd.DataFrame([
    {'model': 'Seasonal-naive baseline', **naive_metrics},
    {'model': 'Gradient Boosting', **gb_metrics},
    {'model': 'Ensemble (50/50 average)', **ensemble_avg_metrics},
    {'model': 'Ensemble (30% naive / 70% GB)', **ensemble_weighted_metrics},
])
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#9fb3c8', '#2d6a9f', '#7fb069', '#1b8a3a']
bars = ax.bar(comparison['model'], comparison['mae'], color=colors)
ax.set_title('All four models — MAE comparison (lower is better)', fontweight='bold')
ax.set_ylabel('MAE (°C)')
plt.xticks(rotation=20, ha='right')
for bar, val in zip(bars, comparison['mae']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f'{val:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()
best = comparison.loc[comparison['mae'].idxmin()]
print(f"Best model: {best['model']} (MAE {best['mae']:.3f}°C)")
print('The weighted ensemble beats every individual model.')

## 8. Seasonal Pattern Analysis (D14)

Framed honestly: this dataset spans about two years, enough to compare the same season across two consecutive years, but **not** enough for multi-decade climate trend claims. June–December has full coverage in both 2024 and 2025.

In [ ]:
df_clean['year'] = df_clean['last_updated'].dt.year
df_clean['month'] = df_clean['last_updated'].dt.month

overlap_months = list(range(6, 13))
yoy = (
    df_clean[df_clean['month'].isin(overlap_months) & df_clean['year'].isin([2024, 2025])]
    .groupby(['year', 'month'])['temperature_celsius'].mean().unstack(0)
)
yoy

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
yoy.plot(ax=ax, marker='o', color=['#9fb3c8', '#2d6a9f'])
ax.set_title('Mean global temperature by month — 2024 vs 2025 (overlapping months only)', fontweight='bold')
ax.set_ylabel('Temperature (°C)')
ax.legend(title='Year')
plt.tight_layout()
plt.show()
diff = (yoy[2025] - yoy[2024]).mean()
print(f'Mean difference (2025 - 2024): {diff:+.2f}°C')
print('Single year-over-year comparison, not a climate trend — two years is')
print('nowhere near enough to separate warming from ordinary variability.')

## 9. Environmental Impact — Air Quality vs. Weather (D15)

In [ ]:
env_cols = ['temperature_celsius', 'humidity', 'wind_kph', 'air_quality_PM2.5', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide']
env_corr = df_clean[env_cols].corr()

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(env_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, linewidths=0.5)
ax.set_title('Weather vs. air quality correlations', fontweight='bold')
plt.tight_layout()
plt.show()
temp_pm25_corr = df_clean['temperature_celsius'].corr(df_clean['air_quality_PM2.5'])
print(f'Temperature vs PM2.5 correlation: {temp_pm25_corr:.3f} (weak positive, not strongly causal)')

## 10. Geographic Patterns (D17, D18)

In [ ]:
country_stats = (
    df_clean.groupby('country_clean')
    .agg(mean_temp=('temperature_celsius', 'mean'), mean_pm25=('air_quality_PM2.5', 'mean'), n=('temperature_celsius', 'count'))
    .reset_index().sort_values('mean_temp', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].barh(country_stats.head(10)['country_clean'], country_stats.head(10)['mean_temp'], color='#e07b54')
axes[0].set_title('10 warmest countries')
axes[0].invert_yaxis()
axes[1].barh(country_stats.tail(10)['country_clean'], country_stats.tail(10)['mean_temp'], color='#2d6a9f')
axes[1].set_title('10 coldest countries')
axes[1].invert_yaxis()
plt.suptitle('Country-level temperature extremes (post-canonicalization)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
loc_stats = df_clean.groupby('location_name').agg(lat=('latitude', 'first'), mean_temp=('temperature_celsius', 'mean'), mean_pm25=('air_quality_PM2.5', 'mean')).reset_index()

fig, ax = plt.subplots(figsize=(11, 6))
sc = ax.scatter(loc_stats['lat'], loc_stats['mean_temp'], c=loc_stats['mean_pm25'], cmap='YlOrRd', s=55, alpha=0.85)
plt.colorbar(sc, ax=ax, label='Mean PM2.5')
ax.axvline(0, color='grey', linestyle='--', alpha=0.5, label='Equator')
ax.set_xlabel('Latitude')
ax.set_ylabel('Mean temperature (°C)')
ax.set_title('Latitude vs. temperature, coloured by air quality', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Per-City Forecast Comparison

Three contrasting climates: tropical, temperate, polar — to see how seasonal swing differs by location.

In [ ]:
candidate_cities = ['Singapore', 'London', 'Reykjavik']
available = [c for c in candidate_cities if c in df_clean['location_name'].unique()]
print('Using cities:', available)

fig, axes = plt.subplots(1, len(available), figsize=(6*len(available), 5))
if len(available) == 1:
    axes = [axes]
for ax, city in zip(axes, available):
    city_data = df_clean[df_clean['location_name'] == city].sort_values('last_updated')
    ax.plot(city_data['last_updated'], city_data['temperature_celsius'], color='#2d6a9f', linewidth=0.8, alpha=0.7)
    ax.set_title(city)
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Temperature time series — contrasting climates', fontweight='bold')
plt.tight_layout()
plt.show()
for city in available:
    city_data = df_clean[df_clean['location_name'] == city]
    seasonal_range = city_data['temperature_celsius'].max() - city_data['temperature_celsius'].min()
    print(f'{city}: seasonal range = {seasonal_range:.1f}°C')

## 12. Model Artifact Export

In [ ]:
import joblib, os

prod_model, prod_features = train_gradient_boosting(df_clean)
prod_baselines = build_location_baselines(df_clean)
known_coords = df_clean[['latitude','longitude']].round(2).drop_duplicates()

os.makedirs('../models', exist_ok=True)
joblib.dump({'model': prod_model, 'features': prod_features}, '../models/forecast_model.joblib')
joblib.dump(prod_baselines, '../models/location_baselines.joblib')
joblib.dump(known_coords, '../models/known_coverage_coords.joblib')

print('Artifacts saved to ../models/ — loaded at FastAPI startup by app/ml/loader.py')
print('and served via /forecast and /anomaly-check endpoints.')

## Summary

| Item | Result |
|---|---|
| Dataset size | 147,930 rows × 41 columns |
| Date range | 2024-05-16 → 2026-06-17 |
| Locations | 268 cities, 195 countries (after canonicalization) |
| Missing values | Zero |
| Physically implausible readings | 4 (flagged, not silently dropped) |
| Baseline MAE | 4.017°C |
| Gradient Boosting MAE | 3.765°C |
| Best ensemble MAE | 3.674°C (30% naive / 70% GB) |
| Year-over-year seasonal pattern | reported, not claimed as climate trend |

**The trained model artifacts are served live through the Atlas FastAPI backend.** This notebook is the upstream producer of the intelligence that powers the live API — not a standalone deliverable.

---
*Atlas Weather Intelligence Platform — built for PM Accelerator AI Engineer Internship*